# Is Brooklyn's higher return rate a provider-mix effect?

Direct standardization of each borough's return rate to the pooled provider mix. Provider names are pseudonymized in the saved output.

In [ ]:
# Run from anywhere inside the repo: find the root, make it the working
# directory, and put it on sys.path so `pipeline/`, `descriptives/` and
# `paths.py` import cleanly. Data paths below are relative to the root.
import os, sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'paths.py').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
import sys
import pandas as pd, numpy as np
from descriptives import cohort_table as ct

panel  = pd.read_csv(ct.PANEL)
lookup = pd.read_csv(ct.LOOKUP, usecols=[ct.ID, "Year", "service_option"])
df     = ct.build_outcome(panel, lookup)
len(df), df.returned.mean()

(130417, np.float64(0.5939793125129392))

In [10]:
# 1. drop the largest Brooklyn provider
#    (rates only: printing its row count would reveal the provider's size)
bk = df[df.borough.eq("Brooklyn")]
TOP = bk.provider.value_counts().idxmax()
print(f"Brooklyn all:        {bk.returned.mean():.3f}")
print(f"Brooklyn minus top:  {bk[~bk.provider.eq(TOP)].returned.mean():.3f}")
print(f"Pooled:              {df.returned.mean():.3f}")

print('\n')

# 2. do providers span boroughs?
#    Only the summary share is printed. A provider-by-borough table would
#    fingerprint individual providers, even under pseudonyms.
ct_pb = pd.crosstab(df.provider, df.borough)
print("providers >90% in one borough:",
      (ct_pb.div(ct_pb.sum(1), axis=0).max(axis=1) > .9).mean().round(2))

print('\n')

# 3. direct standardization to the pooled provider mix
d = df.assign(provider=df.provider.fillna("(missing)"))

w    = d.groupby("provider").size() / len(d)          # sums to 1
rate = d.groupby(["borough", "provider"]).returned.mean().unstack()
present = rate.notna()

num = (rate.fillna(0) * w).sum(axis=1)
den = (present * w).sum(axis=1)
adj = num / den

out = pd.DataFrame({"crude": d.groupby("borough").returned.mean(),
                    "adjusted": adj,
                    "coverage": den,
                    "n": d.groupby("borough").size()})
out["shift"] = (out.adjusted - out.crude).round(3)

# sanity: size-weighted adjusted should land near the pooled rate
print(out.round(3))
print("check:", np.average(out.adjusted, weights=out.n).round(3), "vs", d.returned.mean().round(3))


Brooklyn all:        0.628
Brooklyn minus top:  0.589
Pooled:              0.594


providers >90% in one borough: 0.52


               crude  adjusted  coverage      n  shift
borough                                               
Bronx          0.566     0.547     1.000  27362 -0.020
Brooklyn       0.628     0.595     1.000  57895 -0.033
Manhattan      0.568     0.551     0.995  11402 -0.018
Queens         0.565     0.554     0.991  28579 -0.011
Staten Island  0.576     0.602     0.900   5179  0.025
check: 0.572 vs 0.594


Brooklyn's higher rate is partly provider composition: setting aside its single largest provider brings Brooklyn from 62.8% to 58.9%, in line with the pooled 59.4%. But standardizing every borough to the same provider mix only closes about a third of the gap, so most of it is something else.